In [ ]:
##import team EPA data from event code
## Run only when event matches are active. Recommended to run on device plugged into power in pit.

!pip install statbotics==3.0.0
!pip install gspread pandas -q

from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
import statbotics
import pandas as pd
import requests
import time

creds, _ = default()
gc = gspread.authorize(creds)

sh = gc.open("Worlds 2026")
ws = sh.worksheet("Data")
ws_mode = sh.worksheet("MODE")  # event code in C2

sb = statbotics.Statbotics()

TBA_API_KEY = "FpN1CQK6xbMb4zxYq0NOMQv07h7jNjTtvnpPvkMvRk4LruX4jYYzponLWJtLgtik"

while True:
    print("Run at", time.strftime("%H:%M:%S"))

    # --- 0. Read event code from MODE!C2 ---
    cell = ws_mode.acell("C2").value
    if not cell or not cell.strip():
        print("⚠️ MODE!C2 is empty; skipping EPA & ranks update. Enter an event code (e.g., 2026nytr).")
        time.sleep(600)
        continue

    EVENT_KEY = cell.strip()
    print("Using event code from MODE!C2:", EVENT_KEY)

    # --- 1. Pull team numbers from TBA event ---
    teams_url = f"https://www.thebluealliance.com/api/v3/event/{EVENT_KEY}/teams/simple"
    headers = {"X-TBA-Auth-Key": TBA_API_KEY}

    print("Fetching team numbers from TBA event:", EVENT_KEY)
    r = requests.get(teams_url, headers=headers)
    r.raise_for_status()
    teams_data = r.json()

    team_numbers = sorted([int(t["team_number"]) for t in teams_data])
    print("Team numbers:", team_numbers)

    target_teams = set(team_numbers)

    # --- 2a. Statbotics EPA data ---
    rows = []
    for team in team_numbers:
        try:
            data = sb.get_team_year(team, 2026)
            epa_data = data.get("epa", {})
            epa_breakdown = epa_data.get("breakdown", {})

            rows.append({
                "team_number": data.get("team", team),
                "total_epa": epa_data.get("total_points", {}).get("mean"),
                "auto_epa": epa_breakdown.get("auto_points"),
                "teleop_epa": epa_breakdown.get("teleop_points"),
                "endgame_epa": epa_breakdown.get("endgame_points"),
            })
        except Exception as e:
            print(f"⚠️ Error for team {team}: {e}")
            rows.append({
                "team_number": team,
                "total_epa": None,
                "auto_epa": None,
                "teleop_epa": None,
                "endgame_epa": None,
            })

    df = pd.DataFrame(rows)
    ws.clear()
    ws.update([df.columns.tolist()] + df.values.tolist())
    print("✅ EPAs updated for", len(team_numbers), "teams.")

    # --- 2b. TBA qual ranks for same event (from MODE!C2) ---
    rank_by_team = {}
    try:
        time.sleep(0.2)
        rankings_url = f"https://www.thebluealliance.com/api/v3/event/{EVENT_KEY}/rankings"
        r = requests.get(rankings_url, headers=headers)
        r.raise_for_status()
        rankings_data = r.json()

        for rank_entry in rankings_data.get("rankings", []):
            team_key = rank_entry.get("team_key", "")
            if team_key.startswith("frc"):
                team_num = int(team_key.replace("frc", ""))
                if team_num in target_teams:
                    rank = rank_entry.get("rank", "")
                    rank_by_team[team_num] = rank

        print("✅ Fetched qual ranks:", rank_by_team)
    except Exception as e:
        print("❌ Error fetching TBA rankings:", e)

    # --- 3. Write qual ranks to column F (next to team numbers in column A) ---
    all_cells = ws.get("A2:A")
    updates = []

    for i, cell in enumerate(all_cells, start=2):
        if not cell or not str(cell[0]).strip():
            updates.append({"range": f"F{i}", "values": [[""]]})
            continue
        try:
            team_num = int(cell[0])
            qual_rank = rank_by_team.get(team_num, "")
            updates.append({"range": f"F{i}", "values": [[qual_rank]]})
        except (ValueError, IndexError):
            updates.append({"range": f"F{i}", "values": [[""]]})

    if updates:
        ws.batch_update(updates)
        print("✅ Qual ranks updated.")

    time.sleep(600)  # adjust as needed

In [ ]:
## import qual matches from event code
## Run Once post Qual Match announcement on TBA

!pip install gspread pandas -q

from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
import pandas as pd
import requests
import time

creds, _ = default()
gc = gspread.authorize(creds)

# Open the sheet
sh = gc.open("Worlds 2026")
ws_qual = sh.worksheet("qual_matches")   # qual‑match teams
ws_mode = sh.worksheet("MODE")           # event code in B2

TBA_API_KEY = "FpN1CQK6xbMb4zxYq0NOMQv07h7jNjTtvnpPvkMvRk4LruX4jYYzponLWJtLgtik"

while True:
    print("Run at", time.strftime("%H:%M:%S"))

    # Read event code from MODE!B2
    cell = ws_mode.acell("C2").value
    if not cell or not cell.strip():
        print("⚠️ MODE!B2 is empty; skipping qual‑match update. Enter an event code (e.g., 2026cmptx).")
        time.sleep(10)
        continue

    EVENT_KEY = cell.strip()
    print("Using event code from MODE!B2:", EVENT_KEY)

    matches_url = f"https://www.thebluealliance.com/api/v3/event/{EVENT_KEY}/matches"
    headers = {"X-TBA-Auth-Key": TBA_API_KEY}

    r = requests.get(matches_url, headers=headers)
    r.raise_for_status()
    matches = r.json()

    qual_rows = []
    for match in matches:
        if match["comp_level"] != "qm":  # only qual matches
            continue
        mid = match.get("match_number")
        if not mid:
            continue

        red = match["alliances"]["red"]["team_keys"]
        blue = match["alliances"]["blue"]["team_keys"]

        try:
            red_nums = [int(t.replace("frc", "")) for t in red]
            blue_nums = [int(t.replace("frc", "")) for t in blue]

            red1, red2, red3 = red_nums
            blue1, blue2, blue3 = blue_nums

            qual_rows.append([
                mid,
                red1, red2, red3,
                blue1, blue2, blue3
            ])
        except (ValueError, IndexError) as e:
            print(f"⚠️ Failed to parse match {mid}: {e}")

    if qual_rows:
        df_qual = pd.DataFrame(
            qual_rows,
            columns=["match", "R1", "R2", "R3", "B1", "B2", "B3"]
        )
        ws_qual.clear()
        ws_qual.update([df_qual.columns.tolist()] + df_qual.values.tolist())
        print("✅ Qual‑match teams written into 'qual_matches' sheet.")
    else:
        print("⚠️ No qual matches found for event:", EVENT_KEY)


In [ ]:
## Creates matches for team defined in mode sheet
## Run Once post Qual Match announcement on TBA

import gspread
import requests

sh = gc.open("Worlds 2026")
ws_mode = sh.worksheet("MODE")

# ========== 1. Read event code (C2) and team number (C3) ==========
event_cell = ws_mode.acell("C2").value
if not event_cell or not event_cell.strip():
    raise ValueError("⚠️ MODE!C2 is empty. Please enter an event code (e.g., 2026nytr).")
EVENT_KEY = event_cell.strip()

team_cell = ws_mode.acell("C3").value
if not team_cell or not team_cell.strip():
    raise ValueError("⚠️ MODE!C3 is empty. Please enter a team number (e.g., 263).")
SCOUT_TEAM = int(team_cell.strip())

print("Using event code from MODE!C2:", EVENT_KEY)
print("Scouting team from MODE!C3:", SCOUT_TEAM)

TBA_API_KEY = "FpN1CQK6xbMb4zxYq0NOMQv07h7jNjTtvnpPvkMvRk4LruX4jYYzponLWJtLgtik"
matches_url = f"https://www.thebluealliance.com/api/v3/event/{EVENT_KEY}/matches"
headers = {"X-TBA-Auth-Key": TBA_API_KEY}

# ========== 2. Get ONLY qualification matches where the team plays ==========
r = requests.get(matches_url, headers=headers)
r.raise_for_status()
matches = r.json()

TEAM_KEY = f"frc{SCOUT_TEAM}"
qual_match_nums = []

for match in matches:
    # Only qual matches
    if match.get("comp_level") != "qm":
        continue
    mid = match.get("match_number")
    if not mid:
        continue

    # Check Red and Blue
    for alliance in ["red", "blue"]:
        if TEAM_KEY in match["alliances"][alliance]["team_keys"]:
            qual_match_nums.append(str(mid))
            break

qual_match_nums = sorted(qual_match_nums, key=int)  # numeric order
print("Qual match numbers for team:", qual_match_nums)

# ========== 3. Template: MATCH MASTER DOC ==========
TEMPLATE_SHEET = "MATCH MASTER DOC"
try:
    template_ws = sh.worksheet(TEMPLATE_SHEET)
except gspread.exceptions.WorksheetNotFound:
    raise ValueError(f"Sheet '{TEMPLATE_SHEET}' not found. Please ensure it exists.")

all_titles = [ws.title for ws in sh.worksheets()]

# ========== 4. Create one sheet per qual match ==========
for mid in qual_match_nums:
    sheet_name = f"QM{mid}"  # e.g., QM1, QM2, ...

    if sheet_name in all_titles:
        print(f"✅ Sheet '{sheet_name}' already exists.")
        continue

    # Copy MATCH MASTER DOC
    resp = template_ws.copy_to(sh.id)
    temp_title = resp["title"]

    sh.fetch_sheet_metadata()
    new_ws = sh.worksheet(temp_title)

    # Rename sheet
    new_ws.update_title(sheet_name)
    print(f"✅ Created qual‑match sheet '{sheet_name}' for team {SCOUT_TEAM}.")

    # Write match number into J2 (no leading ')
    new_ws.update("J2", [[mid]], value_input_option="USER_ENTERED")
    print(f"   ➤ Wrote '{mid}' into J2 of sheet '{sheet_name}'.")

In [ ]:
##Create Spreadsheet for each team at an event from event code set in mode.
## Run Once when Teams at a given comp are confimed. Suggest running just before beginging pit scouting. Takes about 1 Second per team.

import gspread
import requests

# Re‑use existing gc from auth
sh = gc.open("Worlds 2026")

MODE_SHEET = "MODE"
TEMPLATE_SHEET = "TEAM MASTER DOC"

# ========== 1. Read event code from MODE!C2 ==========
ws_mode = sh.worksheet(MODE_SHEET)
cell = ws_mode.acell("C2").value

if not cell or not cell.strip():
    raise ValueError("⚠️ MODE!C2 is empty. Please enter an event code (e.g., 2026nytr).")

EVENT_KEY = cell.strip()
print("Using event code from MODE!C2:", EVENT_KEY)

# ========== 2. Get team numbers from TBA ==========
TBA_API_KEY = "FpN1CQK6xbMb4zxYq0NOMQv07h7jNjTtvnpPvkMvRk4LruX4jYYzponLWJtLgtik"
teams_url = f"https://www.thebluealliance.com/api/v3/event/{EVENT_KEY}/teams/simple"
headers = {"X-TBA-Auth-Key": TBA_API_KEY}

r = requests.get(teams_url, headers=headers)
r.raise_for_status()
teams_data = r.json()

team_numbers = sorted([int(t["team_number"]) for t in teams_data])
print("Team numbers:", team_numbers)

# ========== 3. Create a sheet per team from TEAM MASTER DOC ==========
try:
    template_ws = sh.worksheet(TEMPLATE_SHEET)
except gspread.exceptions.WorksheetNotFound:
    raise ValueError(f"Sheet '{TEMPLATE_SHEET}' not found. Please ensure it exists.")

all_titles = [ws.title for ws in sh.worksheets()]

for team in team_numbers:
    sheet_name = str(team)

    if sheet_name in all_titles:
        print(f"✅ Sheet '{sheet_name}' already exists.")
        continue

    # Copy TEAM MASTER DOC
    resp = template_ws.copy_to(sh.id)
    temp_title = resp["title"]  # e.g. "TEAM MASTER DOC (1)"

    # Reload metadata and get the new sheet
    sh.fetch_sheet_metadata()
    new_ws = sh.worksheet(temp_title)

    # Rename to team number
    new_ws.update_title(sheet_name)
    print(f"✅ Created sheet '{sheet_name}' as copy of '{TEMPLATE_SHEET}'.")

    # Write team number into C2 (no leading ')
    new_ws.update("C2", [[sheet_name]], value_input_option="USER_ENTERED")
    print(f"   ➤ Wrote '{sheet_name}' into C2 of sheet '{sheet_name}'.")